## Information about the data

Q1: What are the observational units in the first data set \(the one you will use as the left\-hand one in the merge\)?\
A1: I'm using video game sales and video games critic scores datasets
Sales # Rank, Name, Platform, Year, Genre, Publisher, Sales per regions, other sales, Global sales

Q2: What are the observational units in the second data set \(the one you will use as the right\-hand one in the merge\)?\
A2: name, platform, release date, score, user score, developer, genre, players(some missing), # of critics, and metacritic users

Q3: What will the observational units be in the merged data set you expect to have as a result of this assignment?\
A3: sales vs review scores

Q4: Will you be doing a left, right, inner, or outer join?  Don't just state the answer in one word, but give a reason why your answer makes sense for the two data sets you have \(and possibly also your goals for the merge, if appropriate\)\.\
A4: I'm doing an inner merge where I merge the critic scores dataset into the sales dataset

Q5: How many rows and columns do you expect to have in the merged data set, and what is your reason for each of those numbers?\
A5: There will be less than 12k rows and 14 columns. Comparing the two datasets, the critics set has about 12k entries in it for reviewed games where the sales has over 16k. After running a match with all games in both sets, I will remove all other games from the sets so as to simplify the data, so at the most my merged set will have 12k rows. I'm using 14 columns so as to compare sales across regions, by total sales, genre, scores(both critics and users), # of critics, and # of players if applicable for popularity measures

Disclaimer, the data I'm using after cleaning is about 40% of the total entries. This is enough due to there being over 11000 unique entries that would take too many hours of manual checking to fix
Additionally, when trying to fuzzy match the entries that don't perfect match, the names of some games are matching, but often the numbers are not matching so sequels are being matched to prequels and vice versa
The data I'm using is not a fully random set, if you really want to make sure everything is perfect go through the datasets yourself be my guest. Console names, sequel numbers, and other game notations may make this a lot more complicated should you try to do this.

Importing necessary libraries

In [6]:
import pandas as pd
from difflib import get_close_matches

## Import both data sets

Creating the dataframes I will be merging after cleaning further down

In [7]:
# sales numbers in millions, review scores out of 10
sales = pd.read_csv("vgsales.csv")
reviews = pd.read_csv("games-data.csv")
# sales.head()
# reviews.head()


## Merge the two data sets

Looking for all directly matching game titles by name and adding them to a list for reference when merging

In [8]:
matches = sales.loc[sales["Name"].isin(reviews["name"]), "Name"].dropna().tolist()
count = len(matches)
print(count)
matches

8183


['Wii Sports',
 'Mario Kart Wii',
 'Wii Sports Resort',
 'Tetris',
 'New Super Mario Bros.',
 'Wii Play',
 'New Super Mario Bros. Wii',
 'Mario Kart DS',
 'Wii Fit',
 'Wii Fit Plus',
 'Kinect Adventures!',
 'Grand Theft Auto V',
 'Grand Theft Auto: San Andreas',
 'Grand Theft Auto V',
 'Grand Theft Auto: Vice City',
 'Brain Age 2: More Training in Minutes a Day',
 'Gran Turismo 3: A-Spec',
 'Call of Duty: Modern Warfare 3',
 'Call of Duty: Black Ops',
 'Call of Duty: Black Ops II',
 'Call of Duty: Black Ops II',
 'Call of Duty: Modern Warfare 2',
 'Call of Duty: Modern Warfare 3',
 'Grand Theft Auto III',
 'Super Smash Bros. Brawl',
 'Call of Duty: Black Ops',
 'Animal Crossing: Wild World',
 'Mario Kart 7',
 'Halo 3',
 'Grand Theft Auto V',
 'Super Mario 64',
 'Gran Turismo 4',
 'Super Mario Galaxy',
 'Grand Theft Auto IV',
 'Gran Turismo',
 'Super Mario 3D Land',
 'Gran Turismo 5',
 'Call of Duty: Modern Warfare 2',
 'Grand Theft Auto IV',
 'Super Mario 64',
 'Just Dance 3',
 'Call o

Not necessary, but seeing which titles didn't have a match between the datasets

In [9]:
non_matches = reviews.loc[~reviews['name'].isin(sales['Name']), 'name'].unique().tolist()
missing_count = len(non_matches)
print(missing_count)
non_matches

7143


['Red Dead Redemption 2',
 'The Legend of Zelda: Breath of the Wild',
 'Super Mario Odyssey',
 'Grand Theft Auto Double Pack',
 "Baldur's Gate II: Shadows of Amn",
 "The Legend of Zelda Collector's Edition",
 'Persona 5 Royal',
 'The Last of Us Remastered',
 'The Legend of Zelda: Ocarina of Time 3D',
 'Celeste',
 "Sid Meier's Civilization II",
 'Super Mario Advance 4: Super Mario Bros. 3',
 'Grim Fandango',
 "Tom Clancy's Splinter Cell Chaos Theory",
 'The Last of Us Part II',
 "Tom Clancy's Splinter Cell Pandora Tomorrow",
 'Divinity: Original Sin II - Definitive Edition',
 'Pac-Man Championship Edition DX',
 'Hades',
 'Half-Life: Alyx',
 'Divinity: Original Sin II',
 'Ori and the Will of the Wisps',
 'Starcraft II: Wings of Liberty',
 'Undertale',
 'Persona 4 Golden',
 'Quake III Arena',
 'Mario Kart Super Circuit',
 'NBA 2K1',
 'Super Smash Bros. Ultimate',
 'Wipeout XL',
 'INSIDE',
 'Flipnote Studio',
 'Forza Horizon 4',
 "Sid Meier's Gettysburg!",
 'Super Street Fighter IV',
 'Wha

Finding the names of all platforms used in both datasets to create a roadmap for conversion to 1 style

In [10]:
r_platforms = reviews["platform"].unique().tolist()
s_platforms = sales["Platform"].unique().tolist()
print(r_platforms)
print(s_platforms)

['Nintendo64', 'PlayStation', 'PlayStation3', 'Dreamcast', 'Xbox360', 'Wii', 'XboxOne', 'Switch', 'PlayStation2', 'PlayStation4', 'GameCube', 'Xbox', 'PC', 'WiiU', 'GameBoyAdvance', '3DS', 'DS', 'PlayStationVita', 'PSP', 'XboxSeriesX', 'PlayStation5', 'Stadia']
['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA', '3DS', 'PS4', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne', 'GC', 'WiiU', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16', '3DO', 'GG', 'PCFX']


Converting the release date column from the video game reviews csv to just be by year for convenience and removing the previous release date column

In [11]:
reviews["Year"] = pd.to_datetime(reviews["r-date"]).dt.year
reviews

,name,platform,r-date,score,user score,developer,genre,players,critics,users,Year
0,The Legend of Zelda: Ocarina of Time,Nintendo64,"November 23, 1998",99,9.1,Nintendo,"Action Adventure,Fantasy",1 Player,22,5749,1998
1,Tony Hawk's Pro Skater 2,PlayStation,"September 20, 2000",98,7.4,NeversoftEntertainment,"Sports,Alternative,Skateboarding",1-2,19,647,2000
2,Grand Theft Auto IV,PlayStation3,"April 29, 2008",98,7.6,RockstarNorth,"Action Adventure,Modern,Modern,Open-World",1 Player,64,3806,2008
3,SoulCalibur,Dreamcast,"September 8, 1999",98,8.5,Namco,"Action,Fighting,3D",1-2,24,324,1999
4,Grand Theft Auto IV,Xbox360,"April 29, 2008",98,7.9,RockstarNorth,"Action Adventure,Modern,Modern,Open-World",1 Player,86,3364,2008
...,...,...,...,...,...,...,...,...,...,...,...
17939,Vroom in the Night Sky,Switch,"April 5, 2017",17,3.1,Poisoft,"Sports,Individual,Biking",No Online Multiplayer,15,105,2017
17940,Leisure Suit Larry: Box Office Bust,PlayStation3,"May 5, 2009",17,1.9,Team17,"Action Adventure,Adventure,Third-Person,Open-W...",No Online Multiplayer,11,45,2009
17941,Yaris,Xbox360,"October 10, 2007",17,4.3,BackboneEntertainment,"Driving,Racing,Arcade,Arcade,Automobile",2 Online,7,129,2007
17942,Ride to Hell: Retribution,PC,"June 24, 2013",16,1.3,"Eutechnyx,DeepSilver","Driving,Modern,Racing,Motorcycle,Motocross,Mod...",No info,9,581,2013


Creating and applying the platform map to the video game reviews dataset for easy merging and matching

In [12]:
platform_map = {'Nintendo64': 'N64', 'PlayStation' : 'PS', 'PlayStation3': 'PS3', 'Dreamcast': 'DC', 'Xbox360': 'X360', 'Wii': 'Wii', 'XboxOne': 'XOne', 'PlayStation2': 'PS2', 'PlayStation4': 'PS4', 'GameCube': 'GC', 'PC': 'PC', 'WiiU': 'WiiU', 'GameBoyAdvance': 'GBA', '3DS': '3DS', 'DS': 'DS', 'PlayStationVita': 'PSV'}
reviews["platform"] = reviews["platform"].map(platform_map)
reviews.dropna(subset=["platform"], inplace=True)
reviews.reset_index(drop=True, inplace=True)
reviews

,name,platform,r-date,score,user score,developer,genre,players,critics,users,Year
0,The Legend of Zelda: Ocarina of Time,N64,"November 23, 1998",99,9.1,Nintendo,"Action Adventure,Fantasy",1 Player,22,5749,1998
1,Tony Hawk's Pro Skater 2,PS,"September 20, 2000",98,7.4,NeversoftEntertainment,"Sports,Alternative,Skateboarding",1-2,19,647,2000
2,Grand Theft Auto IV,PS3,"April 29, 2008",98,7.6,RockstarNorth,"Action Adventure,Modern,Modern,Open-World",1 Player,64,3806,2008
3,SoulCalibur,DC,"September 8, 1999",98,8.5,Namco,"Action,Fighting,3D",1-2,24,324,1999
4,Grand Theft Auto IV,X360,"April 29, 2008",98,7.9,RockstarNorth,"Action Adventure,Modern,Modern,Open-World",1 Player,86,3364,2008
...,...,...,...,...,...,...,...,...,...,...,...
15502,Double Dragon II: Wander of the Dragons,X360,"April 5, 2013",17,1.5,"GravityCorporation,Gravity","Action,Beat-'Em-Up,Beat-'Em-Up,2D",No info,19,58,2013
15503,Leisure Suit Larry: Box Office Bust,PS3,"May 5, 2009",17,1.9,Team17,"Action Adventure,Adventure,Third-Person,Open-W...",No Online Multiplayer,11,45,2009
15504,Yaris,X360,"October 10, 2007",17,4.3,BackboneEntertainment,"Driving,Racing,Arcade,Arcade,Automobile",2 Online,7,129,2007
15505,Ride to Hell: Retribution,PC,"June 24, 2013",16,1.3,"Eutechnyx,DeepSilver","Driving,Modern,Racing,Motorcycle,Motocross,Mod...",No info,9,581,2013


Dropping duplicates in both datasets that may have been created/already exist and resetting the index for merging

In [13]:
reviews.drop_duplicates(subset=["name", "platform", "Year"], inplace=True)
reviews.reset_index(drop=True, inplace=True)
sales.drop_duplicates(subset=["Name", "Platform", "Year"], inplace=True)
sales.reset_index(drop=True, inplace=True)

Merging the two datasets with an inner join by name(game title), platform, and year(release year)\
Did a 1:1 validation to ensure all entries are correctly merged\
Dropping unnecessary columns for analysis later

In [14]:
df_merged = pd.merge(sales, reviews, left_on=["Name", "Platform", "Year"], right_on=["name", "platform", "Year"], validate="1:1", how="inner")
df_merged.drop(columns=["name", "platform", 'r-date', 'genre', 'Rank'], inplace=True)
df_merged.head()

,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,score,user score,developer,players,critics,users
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,76,8.0,Nintendo,No Online Multiplayer,51,429
1,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,82,8.4,Nintendo,Up to 12,73,982
2,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,80,8.1,Nintendo,1-4,73,266
3,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01,89,8.5,Nintendo,No Online Multiplayer,65,639
4,New Super Mario Bros. Wii,Wii,2009.0,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62,87,8.3,Nintendo,No Online Multiplayer,80,767


## Verify the merge

Went through a 1:1 validate to make sure that reviews entries were matched up by at most 1 of the sales entries
See code above 


!Disclaimer! There is potential for more accurate matching if you go through fuzzy matching and check all the games to ensure they are referencing the correct iteration, platform, etc. Requires lots of manual checking to be sure!!! I did not do this, however given the assignment I feel that this was sufficient for accuracy's sake.

## Export your merged data set

Exported merged dataset to be used for analysis

In [ ]:
df_merged.to_csv("Merged Video Game Sales and Reviews.csv")